<a href="https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/final_campaign/VIX_FINAL_OPTUNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VIX Final Optuna v1 — affinage des meilleures configs (4/4)

**Rôle.** Dernier des 4 notebooks de la campagne finale « par acquis de confiance ». Ne reconstruit rien : il recharge le dataset de `VIX_FINAL_FEATURES` **et** les résultats déjà poussés par `VIX_FINAL_ML_SCAN`, sélectionne les configs les plus prometteuses, puis les affine avec Optuna.

**Prérequis** :
- `VIX_FINAL_FEATURES.ipynb` doit avoir tourné et poussé son dataset (`results/vix-final-features`).
- `VIX_FINAL_ML_SCAN.ipynb` doit avoir tourné (même partiellement — au moins quelques milliers de lignes avec ≥3/5 folds par config) et poussé sa progression (`results/vix-final-ml-scan`). Ce notebook peut être relancé plus tard pour affiner des configs supplémentaires si le scan progresse encore.

**Sélection** : top-5 configs par `F1_dir` moyen + top-5 par `F1_UP_FORT` moyen (dédupliquées) parmi celles ayant ≥3 folds valides dans le scan — l'objectif n'est pas de tout affiner (66 000 combinaisons rendraient un Optuna exhaustif ingérable) mais de vérifier si le tuning fin peut faire décoller les configs déjà les meilleures.

**Méthodologie Optuna** (reprise du rapport technique) : `TPESampler`, `MedianPruner`, 100 essais, validation croisée `TimeSeriesSplit` à 3 folds — le tuning se fait sur le train du **dernier fold walk-forward** (le plus de données disponibles). Les hyperparamètres retenus sont ensuite ré-évalués honnêtement sur les 5 folds walk-forward complets, avec la même sélection SHAP et le même resampling par fold que le scan original, pour comparer objectivement au résultat non-tuné.

**Résumable** comme les autres notebooks de la série (reprise + checkpoints après chaque config affinée).

**Comparateurs déjà établis** (walk-forward) : GLOBAL RandomForest h=5j (F1_dir≈0.610±0.025, F1_UP_FORT≈0.359, F1_DOWN_FORT≈0.627), TFT h=5j (F1_dir≈0.559, F1_UP_FORT≈0.118 — infirmé).


In [5]:
import subprocess, sys
pkgs = ['xgboost','lightgbm','catboost','shap','xlsxwriter','imbalanced-learn','optuna','pyarrow']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


Installation OK


In [6]:
import os, time, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_FINAL_OPTUNA'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'flat_thr': 0.003,
    'n_wf_folds': 5,
    'min_train_frac': 0.40,   # doit matcher VIX_FINAL_FEATURES
    'shap_sample': 500,
    'pool_prefilter': 450,
    'top_k_by_f1dir': 5,      # meilleures configs par F1_dir moyen (scan)
    'top_k_by_upfort': 5,     # + meilleures configs par F1_UP_FORT moyen (scan)
    'optuna_trials': 100,     # méthodologie du rapport : 100 trials, TPE, MedianPruner
    'optuna_cv_splits': 3,    # TimeSeriesSplit 3 folds, méthodologie du rapport
    'min_train_rows': 100, 'min_test_rows': 20,
}
TARGET_COL = 'VIX_Amplitude_Class'
GITHUB_REPO = 'LP-D/claude'
FEATURES_BRANCH = 'results/vix-final-features'
SCAN_BRANCH = 'results/vix-final-ml-scan'
RESULTS_BRANCH = 'results/vix-final-optuna'
SCAN_CSV = 'vix_final_ml_scan_results.csv'
RESULTS_CSV = 'vix_final_optuna_results.csv'

print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | top-{CONFIG['top_k_by_f1dir']} par F1_dir + "
      f"top-{CONFIG['top_k_by_upfort']} par F1_UP_FORT | Optuna: {CONFIG['optuna_trials']} trials, "
      f"CV {CONFIG['optuna_cv_splits']} folds")


VIX_FINAL_OPTUNA v1 | top-5 par F1_dir + top-5 par F1_UP_FORT | Optuna: 100 trials, CV 3 folds


In [7]:
# ============================================================
# CHARGEMENT DU DATASET (VIX_FINAL_FEATURES) + DES RÉSULTATS DU SCAN (VIX_FINAL_ML_SCAN)
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

def pull_branch_file(branch, filenames, dest_dir="."):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_REPO}.git"
    workdir = f"/content/_pull_{branch.replace('/', '_')}"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", branch, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(f"Impossible de récupérer '{branch}' : {clone.stderr[-500:]}")
    for fn in filenames:
        src = f"{workdir}/{fn}"
        if os.path.exists(src):
            subprocess.run(["cp", src, f"{dest_dir}/{fn}"], check=True)
        else:
            print(f"  [WARN] {fn} absent de la branche '{branch}'")

if not os.path.exists('vix_final_features.parquet'):
    pull_branch_file(FEATURES_BRANCH, ['vix_final_features.parquet', 'vix_final_features_meta.json'])
    print("[PULL OK] Dataset de features récupéré")
df_features = pd.read_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json') as f:
    meta = json.load(f)
FEATURE_POOL = meta['feature_pool']; VIX_COL = meta['vix_col']; SPX_COL = meta['spx_col']

all_dates = df_features.dropna(how='all').index.sort_values()
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
print(f"Dataset: {df_features.shape} | VIX={VIX_COL} | pool: {len(FEATURE_POOL)} features")

if not os.path.exists(SCAN_CSV):
    pull_branch_file(SCAN_BRANCH, [SCAN_CSV])
    print("[PULL OK] Résultats du scan ML récupérés")
df_scan_all = pd.read_csv(SCAN_CSV)
print(f"Scan ML: {len(df_scan_all)} lignes chargées "
      f"(progression au moment de ce pull — relance VIX_FINAL_ML_SCAN pour aller plus loin)")


Dataset: (6908, 1300) | VIX=IDX_VIX | pool: 1102 features
[PULL OK] Résultats du scan ML récupérés
Scan ML: 25080 lignes chargées (progression au moment de ce pull — relance VIX_FINAL_ML_SCAN pour aller plus loin)


In [8]:
# ============================================================
# SÉLECTION DES MEILLEURES CONFIGS À AFFINER
# ============================================================
ok = df_scan_all.dropna(subset=['F1_dir'])
agg = (ok.groupby(['horizon', 'regime', 'N', 'sampler', 'algo'])
       .agg(F1_dir_mean=('F1_dir', 'mean'), F1_UP_FORT_mean=('F1_UP_FORT', 'mean'),
            F1_DOWN_FORT_mean=('F1_DOWN_FORT', 'mean'), n_folds=('fold', 'nunique'))
       .reset_index())
agg = agg[agg['n_folds'] >= 3].round(4)  # au moins 3/5 folds pour être crédible

if len(agg) == 0:
    raise RuntimeError("Aucune config avec ≥3 folds dans le scan pour l'instant — "
                        "relance VIX_FINAL_ML_SCAN plus longtemps avant d'affiner.")

top_dir = agg.sort_values('F1_dir_mean', ascending=False).head(CONFIG['top_k_by_f1dir'])
top_up = agg.sort_values('F1_UP_FORT_mean', ascending=False).head(CONFIG['top_k_by_upfort'])
TOP_CONFIGS = (pd.concat([top_dir, top_up]).drop_duplicates(
    subset=['horizon', 'regime', 'N', 'sampler', 'algo']).to_dict('records'))

print(f"Configs sélectionnées pour Optuna : {len(TOP_CONFIGS)} "
      f"(top-{CONFIG['top_k_by_f1dir']} F1_dir + top-{CONFIG['top_k_by_upfort']} F1_UP_FORT, dédupliquées)")
for c in TOP_CONFIGS:
    print(f"  h={c['horizon']}j {c['regime']:7s} N={c['N']:2d} {c['sampler']:15s} {c['algo']:16s} "
          f"F1_dir={c['F1_dir_mean']:.3f} UP_FORT={c['F1_UP_FORT_mean']}")


Configs sélectionnées pour Optuna : 10 (top-5 F1_dir + top-5 F1_UP_FORT, dédupliquées)
  h=7j GLOBAL  N=10 SMOTETomek      RandomForest     F1_dir=0.635 UP_FORT=0.4213
  h=10j GLOBAL  N=14 BorderlineSMOTE RandomForest     F1_dir=0.634 UP_FORT=0.37
  h=10j GLOBAL  N=14 ADASYN          RandomForest     F1_dir=0.634 UP_FORT=0.3628
  h=10j GLOBAL  N=14 SMOTETomek      RandomForest     F1_dir=0.632 UP_FORT=0.3464
  h=10j GLOBAL  N=15 ADASYN          RandomForest     F1_dir=0.632 UP_FORT=0.3396
  h=3j STRESS  N= 6 SMOTETomek      RandomForest     F1_dir=0.483 UP_FORT=0.6057
  h=3j STRESS  N= 6 SMOTETomek      XGBoost          F1_dir=0.460 UP_FORT=0.5893
  h=3j STRESS  N= 9 SMOTETomek      XGBoost          F1_dir=0.515 UP_FORT=0.5891
  h=3j STRESS  N= 6 SMOTEENN        LightGBM         F1_dir=0.489 UP_FORT=0.5889
  h=3j STRESS  N= 6 SMOTEENN        GradientBoosting F1_dir=0.490 UP_FORT=0.5881


In [9]:
def build_target(vix_series, horizon, split_idx):
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def metrics(y_true, y_pred):
    dm = {0: 'DOWN', 1: 'DOWN', 2: 'UP', 3: 'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {'F1_4cls': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
         'Acc_dir': round(accuracy_score(yd_t, yd_p), 4),
         'F1_dir': round(f1_score(yd_t, yd_p, average='macro', zero_division=0), 4)}
    ui = [i for i, y in enumerate(y_true) if dm[y] == 'UP']
    di = [i for i, y in enumerate(y_true) if dm[y] == 'DOWN']
    if len(ui) >= 10:
        yt = ['FORT' if y_true[i] == 3 else 'FAIBLE' for i in ui]
        yp = ['FORT' if y_pred[i] == 3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_UP_FORT'] = np.nan
    if len(di) >= 10:
        yt = ['FORT' if y_true[i] == 0 else 'FAIBLE' for i in di]
        yp = ['FORT' if y_pred[i] == 0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_DOWN_FORT'] = np.nan
    return m

def get_clf(algo, p=None):
    p = p or {}
    if algo == 'XGBoost':
        return XGBClassifier(n_estimators=p.get('n_estimators', 200), max_depth=p.get('max_depth', 4),
                             learning_rate=p.get('learning_rate', 0.05), subsample=p.get('subsample', 0.8),
                             colsample_bytree=p.get('colsample_bytree', 0.8),
                             min_child_weight=p.get('min_child_weight', 3), eval_metric='mlogloss',
                             objective='multi:softprob', random_state=SEED, n_jobs=-1, verbosity=0)
    if algo == 'LightGBM':
        return LGBMClassifier(n_estimators=p.get('n_estimators', 200), max_depth=p.get('max_depth', 5),
                              learning_rate=p.get('learning_rate', 0.05), num_leaves=p.get('num_leaves', 31),
                              min_child_samples=p.get('min_child_samples', 10), subsample=p.get('subsample', 0.8),
                              class_weight='balanced', random_state=SEED, verbose=-1, n_jobs=-1)
    if algo == 'RandomForest':
        return RandomForestClassifier(n_estimators=p.get('n_estimators', 200), max_depth=p.get('max_depth', 6),
                                      min_samples_leaf=p.get('min_samples_leaf', 5), class_weight='balanced',
                                      random_state=SEED, n_jobs=-1)
    if algo == 'GradientBoosting':
        return GradientBoostingClassifier(n_estimators=p.get('n_estimators', 200),
                                          learning_rate=p.get('learning_rate', 0.05), max_depth=p.get('max_depth', 4),
                                          min_samples_leaf=p.get('min_samples_leaf', 10),
                                          subsample=p.get('subsample', 0.8), random_state=SEED)
    if algo == 'CatBoost':
        return CatBoostClassifier(iterations=p.get('n_estimators', 200), depth=p.get('max_depth', 6),
                                  learning_rate=p.get('learning_rate', 0.05), loss_function='MultiClass',
                                  auto_class_weights='Balanced', random_state=SEED, verbose=False,
                                  allow_writing_files=False)
    raise ValueError(algo)

def suggest_params(trial, algo):
    if algo in ('XGBoost', 'GradientBoosting', 'LightGBM', 'CatBoost'):
        p = {'n_estimators': trial.suggest_int('n_estimators', 100, 400),
             'max_depth': trial.suggest_int('max_depth', 3, 8),
             'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True)}
        if algo != 'CatBoost':
            p['subsample'] = trial.suggest_float('subsample', 0.6, 1.0)
        if algo == 'XGBoost':
            p['colsample_bytree'] = trial.suggest_float('colsample_bytree', 0.6, 1.0)
            p['min_child_weight'] = trial.suggest_int('min_child_weight', 1, 10)
        if algo == 'LightGBM':
            p['num_leaves'] = trial.suggest_int('num_leaves', 15, 127)
            p['min_child_samples'] = trial.suggest_int('min_child_samples', 5, 50)
        if algo == 'GradientBoosting':
            p['min_samples_leaf'] = trial.suggest_int('min_samples_leaf', 1, 20)
        return p
    if algo == 'RandomForest':
        return {'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20)}
    raise ValueError(algo)

def get_samp(name):
    return {'SMOTE': SMOTE(random_state=SEED),
            'BorderlineSMOTE': BorderlineSMOTE(random_state=SEED, kind='borderline-1'),
            'ADASYN': ADASYN(random_state=SEED),
            'SMOTETomek': SMOTETomek(random_state=SEED),
            'SMOTEENN': SMOTEENN(random_state=SEED)}[name]

def shap_rank(X_tr, y_tr, pool_names, top_n, prefilter):
    nf = X_tr.shape[1]
    if nf > prefilter:
        pf = XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                           eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
        pf.fit(X_tr, y_tr); keep = np.argsort(pf.feature_importances_)[::-1][:prefilter]
    else:
        keep = np.arange(nf)
    Xk = X_tr[:, keep]
    pilot = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    pilot.fit(Xk, y_tr)
    sv = np.abs(np.array(shap.TreeExplainer(pilot).shap_values(Xk[:min(CONFIG['shap_sample'], len(Xk))])))
    nfk = Xk.shape[1]
    feat_axes = [ax for ax in range(sv.ndim) if sv.shape[ax] == nfk]
    if len(feat_axes) == 1:
        arr = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != feat_axes[0]))
    else:
        arr = np.asarray(pilot.feature_importances_)
    order = np.argsort(np.asarray(arr).ravel())[::-1][:top_n]
    return list(keep[order])

print("Helpers OK (build_target, metrics, get_clf/suggest_params paramétrables, get_samp, shap_rank)")


Helpers OK (build_target, metrics, get_clf/suggest_params paramétrables, get_samp, shap_rank)


## Principe : recherche bayésienne d'hyperparamètres (Optuna)

Un grid/random search classique évalue des points indépendamment. **Optuna** utilise un
sampler **TPE** (Tree-structured Parzen Estimator) : il modélise séparément la distribution
des hyperparamètres ayant donné de bons scores vs de mauvais scores, puis échantillonne le
prochain essai dans la zone la plus prometteuse — une recherche qui apprend au fur et à
mesure, plus efficace qu'une grille fixe pour un budget d'essais donné (ici 100).

Le **MedianPruner** arrête tôt les essais dont les scores intermédiaires sont nettement en
dessous de la médiane des essais précédents, économisant du calcul sur des configurations
manifestement mauvaises.

Le score de chaque essai est estimé par **validation croisée `TimeSeriesSplit`** (3 folds) :
contrairement à un k-fold classique, chaque split respecte l'ordre chronologique (le train
est toujours strictement antérieur à la validation), pour ne pas réintroduire de fuite
temporelle au moment même où l'on choisit les hyperparamètres.


In [10]:
# ============================================================
# SYNCHRONISATION DE LA PROGRESSION (même principe que les autres notebooks)
# ============================================================
_PUSH_WORKDIR = "/content/_vix_optuna_push"

def push_progress(label=''):
    if not GITHUB_TOKEN or not os.path.exists(RESULTS_CSV):
        return False
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0: return False
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)
        subprocess.run(["cp", RESULTS_CSV, f"{_PUSH_WORKDIR}/{RESULTS_CSV}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email", "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name", "VIX Final Optuna Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", RESULTS_CSV], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Progression Optuna {label} — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        ok = push.returncode == 0
        if ok: print(f"  [CHECKPOINT PUSHÉ] {label}")
        return ok
    except Exception as e:
        print(f"  [WARN push checkpoint] {e}")
        return False

def pull_progress():
    if os.path.exists(RESULTS_CSV) or not GITHUB_TOKEN:
        return
    url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    exists = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", url, RESULTS_BRANCH],
                            capture_output=True, text=True)
    if exists.returncode != 0:
        print(f"[INFO] Aucune progression antérieure sur '{RESULTS_BRANCH}'.")
        return
    workdir = "/content/_vix_optuna_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", RESULTS_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode == 0 and os.path.exists(f"{workdir}/{RESULTS_CSV}"):
        subprocess.run(["cp", f"{workdir}/{RESULTS_CSV}", "."], check=True)
        print("[PULL OK] Progression Optuna antérieure récupérée.")

pull_progress()


[INFO] Aucune progression antérieure sur 'results/vix-final-optuna'.


In [11]:
# ============================================================
# OPTUNA PAR CONFIG SÉLECTIONNÉE, PUIS RÉ-ÉVALUATION WALK-FORWARD
# 1) Tuning (TPE, MedianPruner, TimeSeriesSplit CV) sur le train du DERNIER
#    fold (le plus de données disponibles = la config "quasi-production").
#    Les features restent celles identifiées par le scan pour cette config
#    (ré-affiner aussi la sélection de features à chaque essai Optuna serait
#    un nouveau facteur combinatoire, hors du périmètre du tuning d'hyperparamètres).
# 2) Les hyperparamètres retenus sont ensuite appliqués sur les 5 folds
#    walk-forward (même sélection SHAP par fold que le scan) pour comparer
#    honnêtement au résultat non-tuné.
# ============================================================
def cfg_key(c): return (c['horizon'], c['regime'], c['N'], c['sampler'], c['algo'])
KEY_COLS = ['horizon', 'regime', 'N', 'sampler', 'algo']

done_keys = set()
if os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0:
    prev = pd.read_csv(RESULTS_CSV, usecols=KEY_COLS)
    done_keys = set(map(tuple, prev.values.tolist()))
    print(f"[REPRISE] {len(done_keys)} configs déjà affinées.")

def save_row(row):
    header = not (os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=header, index=False)
    done_keys.add(tuple(row[c] for c in KEY_COLS))

t0 = time.time()
for cfg in TOP_CONFIGS:
    key = cfg_key(cfg)
    if key in done_keys:
        continue
    h, reg, N, sampler, algo = key
    print(f"\n===== Optuna: h={h}j {reg} N={N} {sampler} {algo} "
          f"(scan: F1_dir={cfg['F1_dir_mean']:.3f} UP_FORT={cfg['F1_UP_FORT_mean']}) =====")

    # --- features + données du dernier fold (le plus de données) pour le tuning ---
    cut_last, nxt_last = FOLD_CUTS[-2], FOLD_CUTS[-1]
    cut_date_last = all_dates[cut_last]
    target_last, reg_r_last, _ = build_target(df_features[VIX_COL], h, cut_last)
    idx_last = target_last.index
    tr_mask_last = np.asarray(idx_last < cut_date_last)
    if reg != 'GLOBAL':
        reg_al_last = reg_r_last.reindex(idx_last).fillna('NORMAL').values
        tr_mask_last = tr_mask_last & (reg_al_last == reg)
    y_tr_last = target_last.values[tr_mask_last].astype(int)
    X_pool_last = df_features[FEATURE_POOL].reindex(idx_last)
    sc_last = RobustScaler()
    X_tr_last = sc_last.fit_transform(np.nan_to_num(X_pool_last.values[tr_mask_last]))
    if len(y_tr_last) < CONFIG['min_train_rows'] * 2:
        print("  [SKIP] Pas assez de données sur le dernier fold pour un tuning CV fiable.")
        save_row({'horizon': h, 'regime': reg, 'N': N, 'sampler': sampler, 'algo': algo,
                  'skipped': 'donnees_insuffisantes'})
        continue
    feat_idx = shap_rank(X_tr_last, y_tr_last, FEATURE_POOL, N, CONFIG['pool_prefilter'])
    Xn = X_tr_last[:, feat_idx]

    def objective(trial):
        params = suggest_params(trial, algo)
        tscv = TimeSeriesSplit(n_splits=CONFIG['optuna_cv_splits'])
        scores = []
        for tri, vai in tscv.split(Xn):
            try:
                Xr, yr = get_samp(sampler).fit_resample(Xn[tri], y_tr_last[tri])
            except Exception:
                Xr, yr = Xn[tri], y_tr_last[tri]
            try:
                clf = get_clf(algo, params); clf.fit(Xr, yr)
                pred = clf.predict(Xn[vai])
                scores.append(f1_score(y_tr_last[vai], pred, average='macro', zero_division=0))
            except Exception:
                scores.append(0.0)
        return float(np.mean(scores)) if scores else 0.0

    study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner(),
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=CONFIG['optuna_trials'], show_progress_bar=False)
    best_params = study.best_params
    print(f"  Meilleur score CV (train dernier fold) = {study.best_value:.4f} | params={best_params}")

    # --- ré-évaluation walk-forward complète, tuné vs non-tuné, même protocole que le scan ---
    rows_tuned = []
    for k in range(CONFIG['n_wf_folds']):
        cut, nxt = FOLD_CUTS[k], FOLD_CUTS[k + 1]
        cut_date, nxt_date = all_dates[cut], all_dates[nxt - 1]
        target, reg_r, _ = build_target(df_features[VIX_COL], h, cut)
        idx = target.index
        tr_mask = np.asarray(idx < cut_date); te_mask = np.asarray((idx >= cut_date) & (idx <= nxt_date))
        if reg != 'GLOBAL':
            reg_al = reg_r.reindex(idx).fillna('NORMAL').values
            tr_mask = tr_mask & (reg_al == reg); te_mask = te_mask & (reg_al == reg)
        y_tr = target.values[tr_mask].astype(int); y_te = target.values[te_mask].astype(int)
        if len(y_tr) < CONFIG['min_train_rows'] or len(y_te) < CONFIG['min_test_rows']:
            continue
        X_pool = df_features[FEATURE_POOL].reindex(idx)
        sc = RobustScaler()
        X_tr = sc.fit_transform(np.nan_to_num(X_pool.values[tr_mask]))
        X_te = sc.transform(np.nan_to_num(X_pool.values[te_mask]))
        fidx = shap_rank(X_tr, y_tr, FEATURE_POOL, N, CONFIG['pool_prefilter'])
        try:
            Xr, yr = get_samp(sampler).fit_resample(X_tr[:, fidx], y_tr)
        except Exception:
            Xr, yr = X_tr[:, fidx], y_tr
        clf = get_clf(algo, best_params); clf.fit(Xr, yr)
        met = metrics(y_te, clf.predict(X_te[:, fidx]))
        rows_tuned.append(met)

    if rows_tuned:
        f1s = [r['F1_dir'] for r in rows_tuned]
        ups = [r['F1_UP_FORT'] for r in rows_tuned if not np.isnan(r['F1_UP_FORT'])]
        dns = [r['F1_DOWN_FORT'] for r in rows_tuned if not np.isnan(r['F1_DOWN_FORT'])]
        tuned_f1dir = float(np.mean(f1s)); tuned_up = float(np.mean(ups)) if ups else np.nan
        tuned_dn = float(np.mean(dns)) if dns else np.nan
    else:
        tuned_f1dir = tuned_up = tuned_dn = np.nan

    save_row({'horizon': h, 'regime': reg, 'N': N, 'sampler': sampler, 'algo': algo, 'skipped': '',
              'scan_F1_dir': cfg['F1_dir_mean'], 'scan_F1_UP_FORT': cfg['F1_UP_FORT_mean'],
              'tuned_F1_dir': round(tuned_f1dir, 4) if tuned_f1dir == tuned_f1dir else np.nan,
              'tuned_F1_UP_FORT': round(tuned_up, 4) if tuned_up == tuned_up else np.nan,
              'tuned_F1_DOWN_FORT': round(tuned_dn, 4) if tuned_dn == tuned_dn else np.nan,
              'best_params': json.dumps(best_params), 'optuna_cv_score': round(study.best_value, 4)})
    print(f"  WF tuné: F1_dir={tuned_f1dir:.4f} (scan non-tuné: {cfg['F1_dir_mean']:.4f}) | "
          f"UP_FORT={tuned_up} (scan: {cfg['F1_UP_FORT_mean']}) | {(time.time()-t0)/60:.1f}min écoulées")
    push_progress(label=f"{len(done_keys)}/{len(TOP_CONFIGS)}")

print(f"\n[OPTUNA] {len(done_keys)}/{len(TOP_CONFIGS)} configs affinées ({(time.time()-t0)/60:.1f}min)")



===== Optuna: h=7j GLOBAL N=10 SMOTETomek RandomForest (scan: F1_dir=0.635 UP_FORT=0.4213) =====
  Meilleur score CV (train dernier fold) = 0.4014 | params={'n_estimators': 291, 'max_depth': 6, 'min_samples_leaf': 1}
  WF tuné: F1_dir=0.6304 (scan non-tuné: 0.6351) | UP_FORT=0.42747999999999997 (scan: 0.4213) | 18.9min écoulées
  [CHECKPOINT PUSHÉ] 1/10

===== Optuna: h=10j GLOBAL N=14 BorderlineSMOTE RandomForest (scan: F1_dir=0.634 UP_FORT=0.37) =====
  Meilleur score CV (train dernier fold) = 0.4044 | params={'n_estimators': 466, 'max_depth': 5, 'min_samples_leaf': 1}
  WF tuné: F1_dir=0.6285 (scan non-tuné: 0.6338) | UP_FORT=0.34774 (scan: 0.37) | 41.3min écoulées
  [CHECKPOINT PUSHÉ] 2/10

===== Optuna: h=10j GLOBAL N=14 ADASYN RandomForest (scan: F1_dir=0.634 UP_FORT=0.3628) =====
  Meilleur score CV (train dernier fold) = 0.4099 | params={'n_estimators': 316, 'max_depth': 7, 'min_samples_leaf': 14}
  WF tuné: F1_dir=0.6355 (scan non-tuné: 0.6336) | UP_FORT=0.36984 (scan: 0.3628

In [12]:
# ============================================================
# SYNTHÈSE : TUNÉ vs NON-TUNÉ (SCAN), COMPARAISON AUX RÉFÉRENCES
# ============================================================
df_opt = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
print(f"Progression: {len(df_opt)}/{len(TOP_CONFIGS)} configs affinées")

REF_RF_GLOBAL = {'F1_dir': 0.610, 'F1_dir_std': 0.025, 'F1_UP_FORT': 0.359, 'F1_DOWN_FORT': 0.627}
REF_TFT = {'F1_dir': 0.559, 'F1_UP_FORT': 0.118}

if len(df_opt):
    ok = df_opt[df_opt['tuned_F1_dir'].notna()].copy()
    if len(ok):
        ok['delta_F1_dir'] = (ok['tuned_F1_dir'] - ok['scan_F1_dir']).round(4)
        ok['delta_F1_UP_FORT'] = (ok['tuned_F1_UP_FORT'] - ok['scan_F1_UP_FORT']).round(4)
        cols = ['horizon', 'regime', 'N', 'sampler', 'algo', 'scan_F1_dir', 'tuned_F1_dir',
                'delta_F1_dir', 'scan_F1_UP_FORT', 'tuned_F1_UP_FORT', 'delta_F1_UP_FORT']
        print("\n### Comparaison tuné vs scan (non-tuné), triée par F1_dir tuné ###")
        print(ok[cols].sort_values('tuned_F1_dir', ascending=False).to_string(index=False))

        n_better = int((ok['delta_F1_dir'] > 0).sum())
        print(f"\n{n_better}/{len(ok)} configs améliorées par Optuna sur F1_dir "
              f"(moyenne du delta: {ok['delta_F1_dir'].mean():+.4f})")

        best_row = ok.sort_values('tuned_F1_dir', ascending=False).iloc[0]
        print(f"\nMeilleure config affinée : h={best_row['horizon']}j {best_row['regime']} "
              f"N={best_row['N']} {best_row['sampler']} {best_row['algo']} "
              f"→ F1_dir={best_row['tuned_F1_dir']:.4f}, F1_UP_FORT={best_row['tuned_F1_UP_FORT']}")

        print("\nRéférences établies (walk-forward) :")
        print(f"  GLOBAL RandomForest h=5j    F1_dir={REF_RF_GLOBAL['F1_dir']:.3f}±{REF_RF_GLOBAL['F1_dir_std']:.3f}  "
              f"F1_UP_FORT={REF_RF_GLOBAL['F1_UP_FORT']:.3f}  F1_DOWN_FORT={REF_RF_GLOBAL['F1_DOWN_FORT']:.3f}")
        print(f"  TFT h=5j (infirmé)          F1_dir={REF_TFT['F1_dir']:.3f}  F1_UP_FORT={REF_TFT['F1_UP_FORT']:.3f}")

        beats_dir = best_row['tuned_F1_dir'] > REF_RF_GLOBAL['F1_dir']
        beats_up = (ok['tuned_F1_UP_FORT'].max() if ok['tuned_F1_UP_FORT'].notna().any() else -1) > REF_RF_GLOBAL['F1_UP_FORT']
        if beats_dir or beats_up:
            print("\n[VERDICT] Au moins une config affinée par Optuna dépasse la référence "
                  "GLOBAL RandomForest établie — à confirmer visuellement dans le détail ci-dessus "
                  "(écart-type des folds, cohérence horizon/régime) avant d'en tirer une conclusion définitive.")
        else:
            print("\n[VERDICT] Aucune config affinée ne dépasse la référence GLOBAL RandomForest "
                  "h=5j walk-forward — cohérent avec le constat répété de cette campagne : "
                  "le gain vient de la simplicité/robustesse du modèle, pas du tuning fin.")
    else:
        print("Aucune ligne exploitable pour l'instant (toutes 'skipped' ou en cours).")

    try:
        with pd.ExcelWriter('VIX_FINAL_OPTUNA_report.xlsx', engine='xlsxwriter') as w:
            if len(ok):
                ok[cols].sort_values('tuned_F1_dir', ascending=False).to_excel(w, 'Tuned_vs_Scan', index=False)
            df_opt.to_excel(w, 'Detail', index=False)
        print("\n[SAVE] VIX_FINAL_OPTUNA_report.xlsx (snapshot à date)")
    except Exception as e:
        print(f"[WARN Export] {e}")
else:
    print("Aucun résultat pour l'instant.")
print(f"\n[NOTE] {RESULTS_CSV} contient le détail complet — le recharger pour reprendre.")


Progression: 10/10 configs affinées

### Comparaison tuné vs scan (non-tuné), triée par F1_dir tuné ###
 horizon regime  N         sampler             algo  scan_F1_dir  tuned_F1_dir  delta_F1_dir  scan_F1_UP_FORT  tuned_F1_UP_FORT  delta_F1_UP_FORT
      10 GLOBAL 14          ADASYN     RandomForest       0.6336        0.6355        0.0019           0.3628            0.3698            0.0070
      10 GLOBAL 15          ADASYN     RandomForest       0.6322        0.6316       -0.0006           0.3396            0.3512            0.0116
       7 GLOBAL 10      SMOTETomek     RandomForest       0.6351        0.6304       -0.0047           0.4213            0.4275            0.0062
      10 GLOBAL 14 BorderlineSMOTE     RandomForest       0.6338        0.6285       -0.0053           0.3700            0.3477           -0.0223
      10 GLOBAL 14      SMOTETomek     RandomForest       0.6324        0.6248       -0.0076           0.3464            0.3161           -0.0303
       3 STRESS  9  

In [13]:
# ============================================================
# PUSH FINAL DU RAPPORT (xlsx) EN PLUS DU CSV DE PROGRESSION
# ============================================================
def push_report_file():
    if not GITHUB_TOKEN or not os.path.exists('VIX_FINAL_OPTUNA_report.xlsx'):
        print("[SKIP] Pas de token ou pas de rapport à pousser.")
        return
    try:
        subprocess.run(["cp", "VIX_FINAL_OPTUNA_report.xlsx", f"{_PUSH_WORKDIR}/VIX_FINAL_OPTUNA_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", "VIX_FINAL_OPTUNA_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Rapport Optuna agrégé — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] VIX_FINAL_OPTUNA_report.xlsx sur '{RESULTS_BRANCH}'")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_progress(label='rapport final')
push_report_file()


  [CHECKPOINT PUSHÉ] rapport final
[PUSH OK] VIX_FINAL_OPTUNA_report.xlsx sur 'results/vix-final-optuna'
